# Phase 1: Chest X-Ray Dataset Preprocessing

This notebook prepares the NIH Chest X-ray dataset metadata for later image-modeling phases. The phase loads the metadata, checks patient-level structure, encodes multi-label disease findings, attaches image file paths, creates patient-safe train/validation/test splits, and exports reusable CSV files.

## 1. Load the metadata

The opening step imports the required libraries, loads `Data_Entry_2017.csv` from the Kaggle NIH Chest X-rays dataset, and previews the first few metadata records. Starting with a preview makes the rest of the notebook easier to follow because the key columns and raw label format are visible before any cleaning begins.


In [1]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]

import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import numpy as np

# Set the path to the metadata file.
file_path = "Data_Entry_2017.csv"

# Load the latest version of the NIH Chest X-ray metadata.
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "nih-chest-xrays/data",
    file_path,
)

print("First 5 records:", df.head())


First 5 records:         Image Index          Finding Labels  Follow-up #  Patient ID  \
0  00000001_000.png            Cardiomegaly            0           1   
1  00000001_001.png  Cardiomegaly|Emphysema            1           1   
2  00000001_002.png   Cardiomegaly|Effusion            2           1   
3  00000002_000.png              No Finding            0           2   
4  00000003_000.png                  Hernia            0           3   

   Patient Age Patient Gender View Position  OriginalImage[Width  Height]  \
0           58              M            PA                 2682     2749   
1           58              M            PA                 2894     2729   
2           58              M            PA                 2500     2048   
3           81              M            PA                 2500     2048   
4           81              F            PA                 2582     2991   

   OriginalImagePixelSpacing[x     y]  Unnamed: 11  
0                        0.143  0.

The preview confirms the main fields available in the metadata: image filename, disease labels, follow-up number, patient ID, age, gender, view position, image dimensions, pixel spacing, and the empty `Unnamed: 11` column.


## 2. Inspect patient-level image counts

The patient-count table groups images by `Patient ID` and sorts patients from the most images to the fewest. This matters because repeated imaging can create leakage if images from the same patient are split across training, validation, or test sets.


In [2]:
(
    df.groupby("Patient ID")["Image Index"]
    .count()
    .reset_index()
    .set_index("Patient ID")
    .sort_values(by="Image Index", ascending=False)
)


,Image Index
Patient ID,
10007,184
13670,173
15530,158
12834,157
13993,143
...,...
30798,1
30797,1
30796,1


The sorted table shows 30,805 unique patients. Patient `10007` has the most images, with 184 records, which confirms that patient-level grouping is important.


A quick check of unique patient IDs confirms the identifier range and the number of patient groups. The patient ID is the grouping key for later splitting, so verifying it early reduces the risk of building splits on the wrong unit.


In [3]:
df["Patient ID"].unique()


array([    1,     2,     3, ..., 30803, 30804, 30805], shape=(30805,))

The output confirms that patient IDs run from `1` through `30805`, matching the patient count found in the grouped table.


## 3. Review dataset structure

Here the schema inspection shows the row count, column names, non-null counts, and data types. This is a basic quality-control step: it tells us which columns are complete, which fields are numeric or categorical, and whether any obvious empty columns need to be removed.


In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112120 entries, 0 to 112119
Data columns (total 12 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Image Index                  112120 non-null  object 
 1   Finding Labels               112120 non-null  object 
 2   Follow-up #                  112120 non-null  int64  
 3   Patient ID                   112120 non-null  int64  
 4   Patient Age                  112120 non-null  int64  
 5   Patient Gender               112120 non-null  object 
 6   View Position                112120 non-null  object 
 7   OriginalImage[Width          112120 non-null  int64  
 8   Height]                      112120 non-null  int64  
 9   OriginalImagePixelSpacing[x  112120 non-null  float64
 10  y]                           112120 non-null  float64
 11  Unnamed: 11                  0 non-null       float64
dtypes: float64(3), int64(5), object(4)
memory usage: 10.3+ MB


The schema shows 112,120 image records across 12 columns. All main metadata fields are complete, while `Unnamed: 11` has zero non-null values.


The numeric summary gives a compact view of ranges, averages, and possible outliers in the metadata. It is especially useful here because clinical metadata can contain unusual values, such as implausible ages, that should be noticed before modeling.


In [5]:
df.describe()


,Follow-up #,Patient ID,Patient Age,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
count,112120.000000,112120.000000,112120.000000,112120.000000,112120.000000,112120.000000,112120.000000,0.0
mean,8.573751,14346.381743,46.901463,2646.078844,2486.438842,0.155649,0.155649,NaN
std,15.406320,8403.876972,16.839923,341.246429,401.268227,0.016174,0.016174,NaN
min,0.000000,1.000000,1.000000,1143.000000,966.000000,0.115000,0.115000,NaN
25%,0.000000,7310.750000,35.000000,2500.000000,2048.000000,0.143000,0.143000,NaN
50%,3.000000,13993.000000,49.000000,2518.000000,2544.000000,0.143000,0.143000,NaN
75%,10.000000,20673.000000,59.000000,2992.000000,2991.000000,0.168000,0.168000,NaN
max,183.000000,30805.000000,414.000000,3827.000000,4715.000000,0.198800,0.198800,NaN


The summary statistics show the distribution of follow-up numbers, patient IDs, ages, image dimensions, and pixel spacing. The maximum patient age is unusually high, so age may need more cleaning if it is used later.


## 4. Clean unused metadata

The empty `Unnamed: 11` column can be removed because it contains no useful information. Dropping it keeps the dataset cleaner and prevents an all-null column from being carried into later preprocessing or exports.


In [6]:
df.drop(columns="Unnamed: 11", inplace=True)


This cleaning step does not print output. Successful execution means `Unnamed: 11` has been dropped from `df`.


A column-name check confirms that the empty column was removed successfully. This small verification step makes the cleaning action explicit and helps catch downstream errors caused by unexpected columns.


In [7]:
df.columns.tolist()


['Image Index',
 'Finding Labels',
 'Follow-up #',
 'Patient ID',
 'Patient Age',
 'Patient Gender',
 'View Position',
 'OriginalImage[Width',
 'Height]',
 'OriginalImagePixelSpacing[x',
 'y]']

The column list confirms that `Unnamed: 11` is gone and that the remaining fields describe the image, patient, view position, dimensions, pixel spacing, and raw findings.


## 5. Examine follow-up patterns

Follow-up counts show how repeated imaging is distributed across patients. The distribution helps explain why patient-aware splitting is necessary instead of treating every image as an independent sample.


In [8]:
df["Follow-up #"].value_counts()


Follow-up #
0      30805
1      13302
2       9189
3       7089
4       5759
       ...  
179        1
180        1
181        1
182        1
183        1
Name: count, Length: 184, dtype: int64

The frequency table shows that follow-up `0` is the most common record type, with 30,805 entries. Follow-up values extend to `183`, so some patients have long imaging histories.


To find the longest follow-up sequence, we select the patient associated with the maximum follow-up number. This highlights the most extreme repeated-imaging case and gives context for how much patient history can appear in the dataset.


In [9]:
df.loc[df["Follow-up #"] == df["Follow-up #"].max(), "Patient ID"]


38264    10007
Name: Patient ID, dtype: int64

The result identifies patient `10007` as the patient with the highest follow-up number, `183`.


Patient `10007` is inspected directly because this patient has the longest follow-up sequence in the dataset. Looking at the actual records makes the leakage risk concrete: the same patient can appear many times, sometimes with changing disease labels.


In [10]:
df[df["Patient ID"] == 10007]


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y]
38081,00010007_000.png,No Finding,0,10007,57,M,PA,2992,2991,0.143,0.143
38082,00010007_001.png,No Finding,1,10007,58,M,AP,2500,2048,0.168,0.168
38083,00010007_002.png,Infiltration,2,10007,58,M,AP,2500,2048,0.168,0.168
38084,00010007_003.png,Edema,3,10007,58,M,AP,2500,2048,0.168,0.168
38085,00010007_004.png,Edema|Effusion|Infiltration,4,10007,58,M,AP,2500,2048,0.168,0.168
...,...,...,...,...,...,...,...,...,...,...,...
38260,00010007_179.png,Atelectasis|Effusion,179,10007,59,M,AP,2500,2048,0.168,0.168
38261,00010007_180.png,Infiltration|Pleural_Thickening,180,10007,59,M,AP,2500,2048,0.168,0.168
38262,00010007_181.png,No Finding,181,10007,59,M,AP,2500,2048,0.168,0.168
38263,00010007_182.png,Consolidation,182,10007,59,M,AP,2500,2048,0.168,0.168


The records for patient `10007` show a long sequence of images with changing labels over time. This reinforces why images from the same patient should stay within the same split.


## 6. Encode multi-label disease findings

The raw `Finding Labels` field is multi-label text, so it needs to be expanded into separate binary disease columns. This transformation is required because a multi-label classifier needs one target indicator per disease rather than one combined string.


In [11]:
dummies_df = df["Finding Labels"].str.get_dummies(sep="|")
dummies_df


,Atelectasis,Cardiomegaly,Consolidation,Edema,Effusion,Emphysema,Fibrosis,Hernia,Infiltration,Mass,No Finding,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax
0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0
2,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112115,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0
112116,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
112117,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
112118,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0


The binary label table shows one column per finding, with `1` indicating that the finding is present and `0` indicating that it is absent. This preserves the multi-label nature of the dataset.


The binary disease labels are joined back to the cleaned metadata, while the raw `Finding Labels` text column and `No Finding` indicator are removed. This produces a cleaner modeling table where the remaining label columns represent positive disease targets.


In [12]:
final_data = pd.concat([df, dummies_df], axis=1)
final_data.drop(columns=["Finding Labels", "No Finding"], inplace=True)
final_data


,Image Index,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],...,Effusion,Emphysema,Fibrosis,Hernia,Infiltration,Mass,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax
0,00000001_000.png,0,1,58,M,PA,2682,2749,0.143,0.143,...,0,0,0,0,0,0,0,0,0,0
1,00000001_001.png,1,1,58,M,PA,2894,2729,0.143,0.143,...,0,1,0,0,0,0,0,0,0,0
2,00000001_002.png,2,1,58,M,PA,2500,2048,0.168,0.168,...,1,0,0,0,0,0,0,0,0,0
3,00000002_000.png,0,2,81,M,PA,2500,2048,0.171,0.171,...,0,0,0,0,0,0,0,0,0,0
4,00000003_000.png,0,3,81,F,PA,2582,2991,0.143,0.143,...,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112115,00030801_001.png,1,30801,39,M,PA,2048,2500,0.168,0.168,...,0,0,0,0,0,1,0,0,1,0
112116,00030802_000.png,0,30802,29,M,PA,2048,2500,0.168,0.168,...,0,0,0,0,0,0,0,0,0,0
112117,00030803_000.png,0,30803,42,F,PA,2048,2500,0.168,0.168,...,0,0,0,0,0,0,0,0,0,0
112118,00030804_000.png,0,30804,30,F,PA,2048,2500,0.168,0.168,...,0,0,0,0,0,0,0,0,0,0


The resulting `final_data` table is the cleaned modeling table before image paths are attached. It keeps metadata fields and disease indicators while removing the raw label string and `No Finding`.


## 7. Attach image file paths

Image files are stored across nested Kaggle folders, so this step discovers all `.png` paths before merging them into the metadata. The metadata alone is not enough for image modeling; later dataset loaders need a reliable path to each X-ray file.


In [13]:
from pathlib import Path

img_dir = Path("/kaggle/input/datasets/organizations/nih-chest-xrays/data")
all_paths = sorted(list(img_dir.glob("**/**/*.png")))


This path-discovery step does not print output. Successful execution means `all_paths` now contains the discovered X-ray image file paths.


Each discovered image path is converted into a filename-to-path mapping table using `Image Index` as the join key. Building this mapping avoids hard-coding folder names and lets the metadata connect to images even though the files are spread across multiple directories.


In [14]:
filenames = [p.name for p in all_paths]

# Create a mapping between each image filename and its full file path.
path_df = pd.DataFrame({"Image Index": filenames, "full_path": all_paths})
path_df.head(10)

print(path_df.iloc[0]["full_path"])


/kaggle/input/datasets/organizations/nih-chest-xrays/data/images_001/images/00000001_000.png


The printed path confirms that image files were found and that each path points to an actual `.png` file inside the Kaggle dataset directory.


With the path mapping ready, the metadata table is merged with full image paths so every row points to its X-ray file. This makes each row self-contained for later training code: labels, patient information, and image location all live together.


In [15]:
final_data = pd.merge(final_data, path_df, on="Image Index", how="inner")
final_data.head(10)


,Image Index,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],...,Emphysema,Fibrosis,Hernia,Infiltration,Mass,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax,full_path
0,00000001_000.png,0,1,58,M,PA,2682,2749,0.143,0.143,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
1,00000001_001.png,1,1,58,M,PA,2894,2729,0.143,0.143,...,1,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
2,00000001_002.png,2,1,58,M,PA,2500,2048,0.168,0.168,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
3,00000002_000.png,0,2,81,M,PA,2500,2048,0.171,0.171,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
4,00000003_000.png,0,3,81,F,PA,2582,2991,0.143,0.143,...,0,0,1,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
5,00000003_001.png,1,3,74,F,PA,2500,2048,0.168,0.168,...,0,0,1,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
6,00000003_002.png,2,3,75,F,PA,2048,2500,0.168,0.168,...,0,0,1,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
7,00000003_003.png,3,3,76,F,PA,2698,2991,0.143,0.143,...,0,0,1,1,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
8,00000003_004.png,4,3,77,F,PA,2500,2048,0.168,0.168,...,0,0,1,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
9,00000003_005.png,5,3,78,F,PA,2686,2991,0.143,0.143,...,0,0,1,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...


In [ ]:
assert final_data["full_path"].notna().all()

The preview shows that `final_data` now includes a `full_path` column. This makes each row ready for image loading in the modeling phase.


## 8. Load the predefined training/validation image list

The official training/validation split is supplied as a text file, so we read the filename list and count how many images it contains. Using the provided split keeps this project aligned with the dataset's intended evaluation structure.


In [16]:
train_images = []
with open("/kaggle/input/datasets/organizations/nih-chest-xrays/data/train_val_list.txt") as file:
    train_images = [line.strip() for line in file]

len(train_images)


86524

The count shows `86,524` image filenames in the official training/validation list.


Those filenames define the training/validation pool, which is extracted from the full modeling table. Filtering by filename ensures that the metadata rows match the official image list exactly.


In [17]:
train_val_data = final_data[final_data["Image Index"].isin(train_images)]
train_val_data


,Image Index,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],...,Emphysema,Fibrosis,Hernia,Infiltration,Mass,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax,full_path
0,00000001_000.png,0,1,58,M,PA,2682,2749,0.143000,0.143000,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
1,00000001_001.png,1,1,58,M,PA,2894,2729,0.143000,0.143000,...,1,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
2,00000001_002.png,2,1,58,M,PA,2500,2048,0.168000,0.168000,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
3,00000002_000.png,0,2,81,M,PA,2500,2048,0.171000,0.171000,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
12,00000004_000.png,0,4,82,M,AP,2500,2048,0.168000,0.168000,...,0,0,0,0,1,1,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112100,00030789_000.png,0,30789,52,F,PA,2021,2021,0.194311,0.194311,...,0,0,0,1,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
112106,00030793_000.png,0,30793,58,F,PA,2021,2021,0.194311,0.194311,...,0,0,0,0,1,1,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
112108,00030795_000.png,0,30795,53,F,PA,2021,2021,0.194311,0.194311,...,0,0,0,0,0,0,1,0,0,/kaggle/input/datasets/organizations/nih-chest...
112114,00030801_000.png,0,30801,39,M,PA,2500,2048,0.168000,0.168000,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...


The displayed `train_val_data` table contains metadata, binary labels, and image paths for records assigned to the training/validation pool.


## 9. Load the predefined test image list

The test set is loaded the same way, using the official `test_list.txt` file. Keeping this split separate protects the final evaluation set from being touched during training or validation decisions.


In [18]:
test_images = []
with open("/kaggle/input/datasets/organizations/nih-chest-xrays/data/test_list.txt") as file:
    test_images = [line.strip() for line in file]

len(test_images)


25596

The count shows `25,596` image filenames in the official test list.


Those test filenames are used to extract the official test records from the modeling table. This gives a clean holdout set with the same columns as the training/validation pool, including labels and image paths.


In [19]:
test_data = final_data[final_data["Image Index"].isin(test_images)]
test_data


,Image Index,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],...,Emphysema,Fibrosis,Hernia,Infiltration,Mass,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax,full_path
4,00000003_000.png,0,3,81,F,PA,2582,2991,0.143,0.143,...,0,0,1,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
5,00000003_001.png,1,3,74,F,PA,2500,2048,0.168,0.168,...,0,0,1,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
6,00000003_002.png,2,3,75,F,PA,2048,2500,0.168,0.168,...,0,0,1,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
7,00000003_003.png,3,3,76,F,PA,2698,2991,0.143,0.143,...,0,0,1,1,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
8,00000003_004.png,4,3,77,F,PA,2500,2048,0.168,0.168,...,0,0,1,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112113,00030800_000.png,0,30800,34,F,PA,2048,2500,0.168,0.168,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
112116,00030802_000.png,0,30802,29,M,PA,2048,2500,0.168,0.168,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
112117,00030803_000.png,0,30803,42,F,PA,2048,2500,0.168,0.168,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
112118,00030804_000.png,0,30804,30,F,PA,2048,2500,0.168,0.168,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...


The displayed `test_data` table contains metadata, binary labels, and image paths for records assigned to the official test split.


## 10. Check for patient leakage between train/validation and test

Before making any additional split, the patient IDs in the training/validation pool and test set are compared for overlap. This check is critical because patient overlap can make model performance look better than it really is.


In [20]:
set(test_data["Patient ID"]).intersection(set(train_val_data["Patient ID"]))


set()

The empty set confirms that no patient IDs overlap between the training/validation pool and the test set. This protects the final evaluation from patient leakage.


The same leakage check is enforced as an assertion so the notebook fails immediately if patient overlap is ever introduced. Assertions turn an important assumption into an executable guardrail instead of leaving it as a visual inspection.


In [21]:
assert set(test_data["Patient ID"]).intersection(set(train_val_data["Patient ID"])) == set()


This assertion has no printed output because it passed. If any patient appeared in both groups, the notebook would stop with an assertion error.


## 11. Create a patient-level validation split

Here we hold out validation patients with `GroupShuffleSplit`, using `Patient ID` as the grouping variable so all images from one patient stay together. This creates a more realistic validation set because the model is evaluated on patients it did not see during training.


In [22]:
from sklearn.model_selection import GroupShuffleSplit

groups = train_val_data["Patient ID"]
gss = GroupShuffleSplit(n_splits=1, test_size=0.12, random_state=42)
train_idx, val_idx = next(gss.split(train_val_data, groups=groups))

val_data = train_val_data.iloc[val_idx]
val_data


,Image Index,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],...,Emphysema,Fibrosis,Hernia,Infiltration,Mass,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax,full_path
23,00000008_000.png,0,8,69,F,PA,2048,2500,0.171000,0.171000,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
24,00000008_001.png,1,8,70,F,PA,2048,2500,0.171000,0.171000,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
25,00000008_002.png,2,8,73,F,PA,2048,2500,0.168000,0.168000,...,0,0,0,0,0,1,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
182,00000038_000.png,0,38,75,M,PA,2992,2991,0.143000,0.143000,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
183,00000038_001.png,1,38,75,M,PA,2992,2991,0.143000,0.143000,...,0,0,0,1,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111973,00030701_002.png,2,30701,53,M,PA,2021,2021,0.194311,0.194311,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
112053,00030754_000.png,0,30754,54,M,PA,2020,2021,0.194314,0.194314,...,0,0,0,1,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
112073,00030772_000.png,0,30772,26,F,AP,3056,2544,0.139000,0.139000,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
112074,00030772_001.png,1,30772,26,F,AP,3056,2544,0.139000,0.139000,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...


The displayed `val_data` table contains the validation records selected by grouped splitting. Since the split is grouped by `Patient ID`, all images for a validation patient stay together.


The remaining grouped indices become the final training subset. This complements the validation holdout while preserving the same patient-level separation rule.


In [23]:
train_data = train_val_data.iloc[train_idx]
train_data


,Image Index,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],...,Emphysema,Fibrosis,Hernia,Infiltration,Mass,Nodule,Pleural_Thickening,Pneumonia,Pneumothorax,full_path
0,00000001_000.png,0,1,58,M,PA,2682,2749,0.143000,0.143000,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
1,00000001_001.png,1,1,58,M,PA,2894,2729,0.143000,0.143000,...,1,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
2,00000001_002.png,2,1,58,M,PA,2500,2048,0.168000,0.168000,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
3,00000002_000.png,0,2,81,M,PA,2500,2048,0.171000,0.171000,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
12,00000004_000.png,0,4,82,M,AP,2500,2048,0.168000,0.168000,...,0,0,0,0,1,1,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112100,00030789_000.png,0,30789,52,F,PA,2021,2021,0.194311,0.194311,...,0,0,0,1,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
112106,00030793_000.png,0,30793,58,F,PA,2021,2021,0.194311,0.194311,...,0,0,0,0,1,1,0,0,0,/kaggle/input/datasets/organizations/nih-chest...
112108,00030795_000.png,0,30795,53,F,PA,2021,2021,0.194311,0.194311,...,0,0,0,0,0,0,1,0,0,/kaggle/input/datasets/organizations/nih-chest...
112114,00030801_000.png,0,30801,39,M,PA,2500,2048,0.168000,0.168000,...,0,0,0,0,0,0,0,0,0,/kaggle/input/datasets/organizations/nih-chest...


The displayed `train_data` table contains the remaining training records after validation patients have been held out.


A second leakage assertion checks that the final training and validation sets do not share any patient IDs. This verification is necessary because the validation split is created after the official train/test split and should be independently checked.


In [24]:
assert set(train_data["Patient ID"]).intersection(set(val_data["Patient ID"])) == set()


This assertion has no printed output because it passed. Passing confirms that the training and validation sets have no patient overlap.


## 12. Define disease label columns

Because `full_path` is now the final column, the disease labels are selected from the first finding column through the column before `full_path`. Defining the label list once keeps the class-balance calculations consistent across train, validation, and test splits.


In [25]:
label_cols = final_data.columns[10:-1]
label_cols


Index(['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion',
       'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule',
       'Pleural_Thickening', 'Pneumonia', 'Pneumothorax'],
      dtype='object')

The output lists the 14 disease label columns used for the split-level class-balance metrics, from `Atelectasis` through `Pneumothorax`.


For the training split, the mean of each binary label gives the positive-label ratio for that disease. These ratios show which classes the model will see often and which ones may need special handling because they are rare.


In [26]:
train_metrics = train_data[label_cols].mean()
train_metrics = pd.DataFrame(train_metrics).reset_index()
train_metrics.columns = ["Finding", "Training Positive Ratio"]
train_metrics


,Finding,Training Positive Ratio
0,Atelectasis,0.094865
1,Cardiomegaly,0.019783
2,Consolidation,0.033326
3,Edema,0.016178
4,Effusion,0.100135
5,Emphysema,0.016978
6,Fibrosis,0.014211
7,Hernia,0.001508
8,Infiltration,0.159812
9,Mass,0.046593


The training metrics table shows each disease's positive rate in the final training split. These values summarize class imbalance for model fitting.


The validation split gets the same positive-label ratio calculation for comparison with training. Similar distributions make validation metrics easier to interpret, while differences warn us about possible split imbalance.


In [27]:
val_metrics = val_data[label_cols].mean()
val_metrics = pd.DataFrame(val_metrics).reset_index()
val_metrics.columns = ["Finding", "Validation Positive Ratio"]
val_metrics


,Finding,Validation Positive Ratio
0,Atelectasis,0.101883
1,Cardiomegaly,0.019323
2,Consolidation,0.030253
3,Edema,0.014053
4,Effusion,0.099639
5,Emphysema,0.012491
6,Fibrosis,0.016297
7,Hernia,0.002537
8,Infiltration,0.155363
9,Mass,0.046843


The validation metrics table shows each disease's positive rate in the validation split. Comparing it with training helps check whether the grouped split preserved similar label frequencies.


The official test split gets the same metric so final evaluation can be interpreted against its label distribution. If the test set has higher disease prevalence, performance may not compare directly to training or validation behavior.


In [28]:
test_metrics = test_data[label_cols].mean()
test_metrics = pd.DataFrame(test_metrics).reset_index()
test_metrics.columns = ["Finding", "Test Positive Ratio"]
test_metrics


,Finding,Test Positive Ratio
0,Atelectasis,0.128106
1,Cardiomegaly,0.041764
2,Consolidation,0.070910
3,Edema,0.036138
4,Effusion,0.181982
5,Emphysema,0.042702
6,Fibrosis,0.016995
7,Hernia,0.003360
8,Infiltration,0.238787
9,Mass,0.068292


The test metrics table shows each disease's positive rate in the official test split. Several findings have higher positive rates in test than in training or validation, which should be considered during evaluation.


## 13. Export prepared data splits

With the splits finalized, the training, validation, and test metadata tables are written to CSV files for later phases. Exporting them freezes the Phase 1 preprocessing decisions so later notebooks can focus on modeling instead of rebuilding the splits.


In [29]:
train_data.to_csv("train_data.csv", index=False)
val_data.to_csv("val_data.csv", index=False)
test_data.to_csv("test_data.csv", index=False)


This export step does not print output. Successful execution creates `train_data.csv`, `val_data.csv`, and `test_data.csv`.


## 14. Export final class-balance summary

Finally, the split-level positive-label ratios are merged into one summary table and saved as a CSV file. This creates a compact reference for class imbalance that can guide loss functions, sampling choices, and evaluation discussion in later phases.


In [31]:
final_metrics = pd.merge(train_metrics, val_metrics, on="Finding", how="inner")
final_metrics = pd.merge(final_metrics, test_metrics, on="Finding", how="inner")
final_metrics.to_csv("final_metrics.csv", index=False)


This final export step does not print output. Successful execution creates `final_metrics.csv`, which compares positive-label ratios across all three splits.


## Phase 1 Preprocessing Summary

- Loaded the NIH Chest X-ray metadata from Kaggle so the project starts from the official image-level records.
- Inspected patient-level image counts because repeated imaging can cause patient leakage if splits are made at the image level only.
- Reviewed the dataset schema, data types, missing values, and numeric summaries to catch empty columns, unexpected types, and possible metadata outliers early.
- Removed the empty `Unnamed: 11` metadata column to keep the exported tables clean.
- Examined follow-up numbers and checked the patient with the longest imaging history to understand the dataset's longitudinal structure.
- Converted multi-label disease strings into separate binary label columns because multi-label modeling needs one target indicator per finding.
- Created the cleaned `final_data` table by combining metadata with disease labels, giving one modeling-ready row per image.
- Found the full image file paths and merged them into the metadata table so later phases can load images directly from each row.
- Loaded the official training/validation and test image lists to stay aligned with the dataset's provided split structure.
- Created official train/validation and test pools using image filenames so the metadata matches the intended image partitions.
- Verified that the train/validation pool and test set have no patient overlap, protecting the final evaluation from leakage.
- Built a grouped validation split so each patient appears in only one split and validation better reflects unseen-patient performance.
- Verified that the final training and validation sets have no patient overlap as a second leakage guardrail.
- Computed positive-label ratios for each disease across train, validation, and test splits to document class imbalance.
- Exported `train_data.csv`, `val_data.csv`, `test_data.csv`, and `final_metrics.csv` so the next phase can begin from stable preprocessing outputs.
